# Olympic Athletes: 120 Years of History

## 1. Dataset Selection
**Dataset:** 120 years of Olympic history: athletes and results



**Source:** Kaggle (heesoo37), originally scraped from sports-reference.com. Covers Athens 1896 to Rio 2016.



**Why I chose it:** I'm interested in sport, and the data is realistically messy. Height, weight and age are missing for many athletes mainly more towards the earlier records, and medal is null for everyone who didn't win one. That makes it a good test of cleaning decisions.

**Questions to explore:**
1. How has female participation changed over time?
2. How have athletes' height and weight changed over time, and do they differ by sport?
3. Which countries have won the most medals, and does this change by era?
4. How does athlete age vary by sport?
5. Do host countries win more medals than usual (home advantage)?

### 2. Initial Data Exploration and Understanding

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

athletes = pd.read_csv("../data/raw/athlete_events.csv")
regions = pd.read_csv("../data/raw/noc_regions.csv")

print("Athletes:", athletes.shape)
print("Regions:", regions.shape)

Athletes: (271116, 15)
Regions: (230, 3)


In [6]:
athletes.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [7]:
print("Columns:", list(athletes.columns))
athletes.info()

Columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  object 
 2   Sex     271116 non-null  object 
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  object 
 7   NOC     271116 non-null  object 
 8   Games   271116 non-null  object 
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  object 
 11  City    271116 non-null  object 
 12  Sport   271116 non-null  object 
 13  Event   271116 non-null  object 
 14  Medal   39783 non-null   object 
dtypes: float64(3), int64(2), object(10)
memory usage: 31.0+ MB


In [12]:
missing = pd.DataFrame({
    "missing": athletes.isna().sum(),
    "percent": (athletes.isna().mean() * 100).round(2)
}).sort_values("percent", ascending=False)
missing

,missing,percent
Medal,231333,85.33
Weight,62875,23.19
Height,60171,22.19
Age,9474,3.49
Sex,0,0.00
ID,0,0.00
Name,0,0.00
Team,0,0.00
NOC,0,0.00
Year,0,0.00


In [13]:
display(athletes.describe().round(2))
print("Duplicate rows:", athletes.duplicated().sum())
print("Years:", athletes["Year"].min(), "to", athletes["Year"].max())
print("Seasons:", athletes["Season"].unique())

,ID,Age,Height,Weight,Year
count,271116.00,261642.00,210945.00,208241.00,271116.00
mean,68248.95,25.56,175.34,70.70,1978.38
std,39022.29,6.39,10.52,14.35,29.88
min,1.00,10.00,127.00,25.00,1896.00
25%,34643.00,21.00,168.00,60.00,1960.00
50%,68205.00,24.00,175.00,70.00,1988.00
75%,102097.25,28.00,183.00,79.00,2002.00
max,135571.00,97.00,226.00,214.00,2016.00


Duplicate rows: 1385
Years: 1896 to 2016
Seasons: ['Summer' 'Winter']


In [14]:
regions.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


## 3. Data Cleaning

### Decision explanations

| Issue | Finding | Decision | Reason |
|---|---|---|---|
| Duplicate rows | 1,385 exact duplicates | Drop them | Same athlete, same event, same Games recorded twice. They would inflate counts and medal totals. |
| Medal nulls (85%) | Null means no medal | Recode to "No Medal" and add a `Won_Medal` flag | These are not missing data. Leaving them as NaN would hide non-medallists from groupby counts. |
| Height / Weight / Age missing | 22% / 23% / 3.5% | Do not impute. Leave as NaN and drop per-question only | Missingness is concentrated in older Games, so filling with averages would invent modern-looking athletes in 1900. Dropping per question keeps the rest of each row usable. |
| Art Competitions | Non-athletic events with odd ages and no body measurements | Remove | Not physical sport, and they distort age and physique analysis. |
| Country names | `NOC` codes only | Merge `noc_regions.csv` to get country names, fall back to `Team` where there is no match | Readable charts. |
| Data types | Text columns stored as object, Age as float | Convert to category / nullable integer | Less memory, clearer intent. |
| Repeated athlete/Games/Event (95) | Mostly 1900 sailing/rowing: one athlete in several boats | Keep | Genuine separate entries, not errors. |
| BMI < 13 (24 rows) | Physically implausible, e.g. 160 cm / 32 kg | Set Height, Weight and BMI to NaN | Can't tell which measurement is wrong. |
| BMI > 45 (50 rows) | Mostly weightlifting, judo, wrestling | Keep | Plausible for super-heavyweight athletes. |
| NOC `SGP` | Team column holds boat names | Set Country to "Singapore" explicitly | The Team fallback gave wrong values. |

In [15]:
df = athletes.copy()   # duplicated incase i mess anything up and would like a extra version and is good for comprison

# Dups
print("Rows before:", len(df))
df = df.drop_duplicates()
print("Rows after: ", len(df), "| removed:", len(athletes) - len(df))

# Same athlete listed twice for the same event at the same olympicGames 
print("Same athlete/Games/Event repeats:", df.duplicated(subset=["ID", "Games", "Event"]).sum())

# Art Competitions - didnt want these involved as im not interested in them and do not demonstratre athletic ability
art = df["Sport"] == "Art Competitions"
print("Art Competitions rows:", art.sum(), "| years:", df.loc[art, "Year"].min(), "-", df.loc[art, "Year"].max())
df = df[~art]
print("Rows now:", len(df))

Rows before: 271116
Rows after:  269731 | removed: 1385
Same athlete/Games/Event repeats: 95
Art Competitions rows: 2263 | years: 1912 - 1948
Rows now: 267468


In [16]:
# Is missingness random cehck
df["Decade"] = df["Year"] // 10 * 10
(df[["Age", "Height", "Weight"]].isna().groupby(df["Decade"]).mean().mul(100).round(1))

,Age,Height,Weight
Decade,,,
1890,42.9,87.9,87.1
1900,30.3,86.8,88.6
1910,3.7,82.0,85.1
1920,19.1,81.3,87.0
1930,3.7,73.1,83.0
1940,15.7,83.3,83.6
1950,5.8,69.1,69.2
1960,1.4,6.1,6.6
1970,0.7,5.2,5.8


In [23]:
#  null means no medal
df["Medal"] = df["Medal"].fillna("No Medal")
df["Medal"] = pd.Categorical(df["Medal"], categories=["No Medal", "Bronze", "Silver", "Gold"], ordered=True) #very useful for searching for people who
df["Won_Medal"] = df["Medal"] != "No Medal"                                                              # ahve medals greater then silver i.e
                                                                                                        # df["Medal"] >= "Silver" works
#  body measurements
df["BMI"] = (df["Weight"] / (df["Height"] / 100) ** 2).round(1)
print("BMI < 13 or > 45:", ((df["BMI"] < 13) | (df["BMI"] > 45)).sum())
df["BMI"].describe().round(1)

BMI < 13 or > 45: 50


count    206800.0
mean         22.8
std           2.9
min          13.3
25%          21.0
50%          22.5
75%          24.2
max          63.9
Name: BMI, dtype: float64

In [18]:
# Country names
assert regions["NOC"].is_unique
n = len(df)
df = df.merge(regions[["NOC", "region"]], on="NOC", how="left")
assert len(df) == n   # merge must not add rows quite useful for later a

unmatched = df.loc[df["region"].isna(), "NOC"].unique()
print("NOC codes with no region:", unmatched)
df["region"] = df["region"].fillna(df["Team"])
df = df.rename(columns={"region": "Country"})

# convert types
for col in df.select_dtypes("object").columns:
    df[col] = df[col].str.strip()

df["Age"] = df["Age"].astype("Int64")
for col in ["Sex", "Season", "Sport", "Country", "NOC"]:
    df[col] = df[col].astype("category")

df.info()

NOC codes with no region: ['SGP' 'ROT' 'TUV']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 267468 entries, 0 to 267467
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   ID         267468 non-null  int64   
 1   Name       267468 non-null  object  
 2   Sex        267468 non-null  category
 3   Age        258533 non-null  Int64   
 4   Height     210885 non-null  float64 
 5   Weight     208185 non-null  float64 
 6   Team       267468 non-null  object  
 7   NOC        267468 non-null  category
 8   Games      267468 non-null  object  
 9   Year       267468 non-null  int64   
 10  Season     267468 non-null  category
 11  City       267468 non-null  object  
 12  Sport      267468 non-null  category
 13  Event      267468 non-null  object  
 14  Medal      267468 non-null  category
 15  Decade     267468 non-null  int64   
 16  Won_Medal  267468 non-null  bool    
 17  BMI        206824 non-null  float64 
 18

In [19]:

from pathlib import Path
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/processed/olympics_clean.csv", index=False)

print(df.shape)

(267468, 19)


In [21]:
# Repeated athlete/Games/Event rows
rep = df[df.duplicated(subset=["ID", "Games", "Event"], keep=False)].sort_values(["ID", "Games", "Event"])
display(rep[["ID", "Name", "Games", "Sport", "Event", "Team", "Age", "Height", "Weight", "Medal"]].head(12))
print("sports", rep["Sport"].value_counts().head(5).to_dict())

#Extreme BMI values were seen as ow as 8.4 which isnt realistic meaning there was some form of entry error going on which needed to be fixed
low = df[df["BMI"] < 13]
high = df[df["BMI"] > 45]
print("BMI < 13:", len(low), "| BMI > 45:", len(high))
display(low[["Name", "Sport", "Year", "Height", "Weight", "BMI"]].head(8))
print("BMI > 45 by sport:", high["Sport"].value_counts().head(6).to_dict())

# 3. Country fallback check
print(df[df["NOC"].isin(["SGP", "ROT", "TUV"])].groupby("NOC", observed=True)["Country"].unique())

,ID,Name,Games,Sport,Event,Team,Age,Height,Weight,Medal
4005,2317,Auguste Albert,1900 Summer,Sailing,Sailing Mixed 1-2 Ton,Martha-1,<NA>,NaN,NaN,Silver
4006,2317,Auguste Albert,1900 Summer,Sailing,Sailing Mixed 1-2 Ton,Martha-1,<NA>,NaN,NaN,Bronze
10840,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),1900 Summer,Sailing,Sailing Mixed 0.5-1 Ton,Pettit-Poucet-6,31,NaN,NaN,No Medal
10841,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),1900 Summer,Sailing,Sailing Mixed 0.5-1 Ton,Carabinier-15,31,NaN,NaN,Gold
15978,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed 0.5-1 Ton,Crabe II-1,28,NaN,NaN,Silver
15979,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed 0.5-1 Ton,Crabe II-4,28,NaN,NaN,Bronze
15980,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed 1-2 Ton,Nina Claire-2,28,NaN,NaN,Bronze
15981,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed 1-2 Ton,Nina Claire-2,28,NaN,NaN,No Medal
15976,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed Open,Crabe II-12,28,NaN,NaN,No Medal
15977,8683,Jacques Baudrier,1900 Summer,Sailing,Sailing Mixed Open,Nina Claire-30,28,NaN,NaN,No Medal


sports {'Sailing': 101, 'Rowing': 28, 'Equestrianism': 8, 'Cycling': 6, 'Polo': 2}
BMI < 13: 24 | BMI > 45: 50


,Name,Sport,Year,Height,Weight,BMI
9764,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
9765,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
9766,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
9767,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
9768,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
9769,Marta Artigas y Masdeu,Gymnastics,1984,160.0,32.0,12.5
64606,Bndicte Evrard,Gymnastics,1992,176.0,38.0,12.3
64607,Bndicte Evrard,Gymnastics,1992,176.0,38.0,12.3


BMI > 45 by sport: {'Weightlifting': 34, 'Judo': 9, 'Wrestling': 5, 'Shooting': 1, 'Boxing': 1, 'Aeronautics': 0}
NOC
ROT    ['Refugee Olympic Athletes']
Categories (212, ...
SGP    ['Singapore', 'June Climene', 'Rika II', 'Sing...
TUV    ['Tuvalu']
Categories (212, object): ['Afghani...
Name: Country, dtype: object


In [22]:
# Singapore Team holds boat names, so set the country specifically fior them
df["Country"] = df["Country"].astype(object)
df.loc[df["NOC"] == "SGP", "Country"] = "Singapore"
df["Country"] = df["Country"].astype("category")
print(df.loc[df["NOC"] == "SGP", "Country"].astype(str).unique())

# Implausible BMI < 13 can't tell which measurement is wrong, so null all three
bad = df["BMI"] < 13
print("Rows nulled:", bad.sum())
df.loc[bad, ["Height", "Weight", "BMI"]] = np.nan

df.to_csv("../data/processed/olympics_clean.csv", index=False)
print(df.shape)

['Singapore']
Rows nulled: 24
(267468, 19)


In [24]:
display(df.loc[df["Name"] == "Marta Artigas y Masdeu", ["Height", "Weight", "BMI"]].head(2))  # after the BMI < 13 CHANGE these are NaN
display(df[["Height", "Weight", "BMI"]].dropna().head(3))
print(df["Medal"].dtype)
print(df["Medal"].cat.categories.tolist())
print(df["Medal"].value_counts())
print(df["Won_Medal"].mean().round(3))

,Height,Weight,BMI
9764,NaN,NaN,NaN
9765,NaN,NaN,NaN


,Height,Weight,BMI
0,180.0,80.0,24.7
1,170.0,60.0,20.8
4,185.0,82.0,24.0


category
['No Medal', 'Bronze', 'Silver', 'Gold']
Medal
No Medal    227852
Gold         13320
Bronze       13244
Silver       13052
Name: count, dtype: int64
0.148


In [25]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/processed/olympics_clean.csv", index=False)